In [2]:
# import json, pathlib

# root = pathlib.Path("data/raw")

# for src in ["privacy_qa", "contractnli", "cuad", "maud"]:
#     n_files = len(list((root / "corpus" / src).glob("*.txt")))
#     with open(root / "benchmarks" / f"{src}.json", encoding="utf-8") as f:
#         n_tests = len(json.load(f)["tests"])
#     print(f"{src}: {n_files} documents, {n_tests} queries")

# with open(root / "benchmarks" / "cuad.json", encoding="utf-8") as f:
#     t = json.load(f)["tests"][0]
# print("\nQUERY:", t["query"][:300])
# print("SNIPPET:", t["snippets"][0])

In [7]:
import pathlib

ROOT = pathlib.Path.cwd()
print(ROOT)
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

root = ROOT / "data" / "raw"
corpus = root / "corpus"

print("project root:", ROOT)
print("benchmarks exists:", (root / "benchmarks").exists())
print("corpus exists:", corpus.exists())

c:\Users\Gandharva Khedekar\OneDrive\Desktop\Pro\LegalDocumentNavigator\notebooks
project root: c:\Users\Gandharva Khedekar\OneDrive\Desktop\Pro\LegalDocumentNavigator
benchmarks exists: True
corpus exists: True


In [8]:
import json, pathlib, collections, statistics

# root = pathlib.Path("../data/raw")
corpus = root / "corpus"

all_files = {p.relative_to(corpus).as_posix() for p in corpus.rglob("*.txt")}
print("corpus files on disk:", len(all_files))

missing = collections.Counter()
mismatch = collections.Counter()
total = collections.Counter()
span_len = collections.defaultdict(list)
text_cache = {}

for src in ["privacy_qa", "contractnli", "cuad", "maud"]:
    with open(root / "benchmarks" / f"{src}.json", encoding="utf-8") as f:
        tests = json.load(f)["tests"]
    for t in tests:
        for s in t["snippets"]:
            total[src] += 1
            fp = s["file_path"]
            if fp not in all_files:
                missing[src] += 1
                continue
            if fp not in text_cache:
                with open(corpus / fp, encoding="utf-8", newline="") as f:
                    text_cache[fp] = f.read()
            a, b = s["span"]
            if text_cache[fp][a:b] != s["answer"]:
                mismatch[src] += 1
            span_len[src].append(b - a)

for src in total:
    L = span_len[src]
    print(f"{src}: snippets={total[src]} missing_file={missing[src]} "
          f"offset_mismatch={mismatch[src]} "
          f"span_len median={statistics.median(L):.0f} max={max(L)}")

corpus files on disk: 698
privacy_qa: snippets=453 missing_file=25 offset_mismatch=0 span_len median=336 max=5198
contractnli: snippets=1389 missing_file=0 offset_mismatch=0 span_len median=261 max=1996
cuad: snippets=6247 missing_file=8 offset_mismatch=0 span_len median=306 max=3546
maud: snippets=2839 missing_file=266 offset_mismatch=2535 span_len median=399 max=8222


In [9]:
import json, difflib, collections

on_disk = sorted(p.relative_to(corpus).as_posix() for p in corpus.rglob("*.txt"))
disk_set = set(on_disk)

def load_tests(src):
    with open(root / "benchmarks" / f"{src}.json", encoding="utf-8") as f:
        return json.load(f)["tests"]

# A) missing files: show the closest real file name
for src in ["privacy_qa", "cuad", "maud"]:
    miss = collections.Counter()
    for t in load_tests(src):
        for s in t["snippets"]:
            if s["file_path"] not in disk_set:
                miss[s["file_path"]] += 1
    print(f"\n{src}: {len(miss)} distinct missing paths")
    for fp, n in list(miss.items())[:5]:
        close = difflib.get_close_matches(fp, on_disk, n=1, cutoff=0.6)
        if close:
            print(f"  {n}x {fp!r}\n     closest on disk: {close[0]!r}")
        else:
            print(f"  {n}x {fp!r}\n     no close match")

# B) MAUD: is the gold text present anywhere in the file?
cache = {}
exact = elsewhere = absent = 0
shifts = collections.Counter()
shown = 0
for t in load_tests("maud"):
    for s in t["snippets"]:
        fp = s["file_path"]
        if fp not in disk_set:
            continue
        if fp not in cache:
            with open(corpus / fp, encoding="utf-8", newline="") as f:
                cache[fp] = f.read()
        text = cache[fp]
        a, b = s["span"]
        if text[a:b] == s["answer"]:
            exact += 1
            continue
        idx = text.find(s["answer"])
        if idx >= 0:
            elsewhere += 1
            shifts[idx - a] += 1
        else:
            absent += 1
            if shown < 2:
                shown += 1
                print(f"\nEXAMPLE span={s['span']} answer_len={len(s['answer'])}")
                print("  text slice:", repr(text[a:a + 120]))
                print("  answer    :", repr(s["answer"][:120]))

print(f"\nMAUD exact={exact} found_elsewhere={elsewhere} absent={absent}")
print("most common offset shifts:", shifts.most_common(5))
print("files containing \\r:", sum("\r" in v for v in cache.values()), "of", len(cache))


privacy_qa: 1 distinct missing paths
  25x 'privacy_qa/TickTick: To Do List with Reminder, Day Planner.txt'
     closest on disk: 'privacy_qa/TickTick_ To Do List with Reminder, Day Planner.txt'

cuad: 1 distinct missing paths
  8x 'cuad/LECLANCHÉ S.A. - JOINT DEVELOPMENT AND MARKETING AGREEMENT.txt'
     closest on disk: 'cuad/LECLANCHÉ S.A. - JOINT DEVELOPMENT AND MARKETING AGREEMENT.txt'

maud: 16 distinct missing paths
  16x 'maud/Cubic Corporation_Investment Group.pdf||Cubic_Corporation_Investment_Group_Amendment No.1.txt'
     no close match
  16x 'maud/Telenav, Inc._Management Led Buyout.pdf||Telenav, Inc._Management Led Buyout Amendment No. 1.txt'
     no close match
  14x 'maud/Capstead_Mortgage_Corp_Benefit_Street_Partners.pdf||Capstead_Mortgage_Corp_Benefit_Street_Partners_Amendment No.1.txt'
     no close match
  17x 'maud/Extended Stay America, Inc._Investment Group.pdf||Extended Stay America, Inc._Investment Group Amendment No. 1.txt'
     no close match
  20x 'maud/Can

In [12]:
import json, re, unicodedata, statistics, collections

def key(p):
    p = unicodedata.normalize("NFC", p)
    return re.sub(r'[<>:"|?*]', "_", p)

disk = {key(p.relative_to(corpus).as_posix()): p for p in corpus.rglob("*.txt")}

def read(path):
    with open(path, encoding="utf-8") as f:   # default newline handling
        return f.read()

cache = {}
for src in ["privacy_qa", "contractnli", "cuad", "maud"]:
    with open(root / "benchmarks" / f"{src}.json", encoding="utf-8") as f:
        tests = json.load(f)["tests"]
    total = missing = mismatch = 0
    lens = []
    for t in tests:
        for s in t["snippets"]:
            total += 1
            k = key(s["file_path"])
            if k not in disk:
                missing += 1
                continue
            if k not in cache:
                cache[k] = read(disk[k])
            a, b = s["span"]
            if cache[k][a:b] != s["answer"]:
                mismatch += 1
            lens.append(b - a)
    print(f"{src}: snippets={total} missing_file={missing} "
          f"offset_mismatch={mismatch} span_len_median={statistics.median(lens):.0f}")

privacy_qa: snippets=453 missing_file=0 offset_mismatch=0 span_len_median=332
contractnli: snippets=1389 missing_file=0 offset_mismatch=0 span_len_median=261
cuad: snippets=6247 missing_file=0 offset_mismatch=0 span_len_median=305
maud: snippets=2839 missing_file=0 offset_mismatch=0 span_len_median=399


In [11]:
import zipfile, re, pathlib

ZIP_PATH = r"C:\Users\Gandharva Khedekar\Downloads\LegalBench-RAG.zip"   # change this

def clean(name):
    return re.sub(r'[<>:"|?*]', "_", name)

n = 0
with zipfile.ZipFile(ZIP_PATH) as z:
    for m in z.infolist():
        if m.is_dir() or "|" not in m.filename:
            continue
        name = m.filename.replace("\\", "/")
        if "corpus/" not in name:
            continue
        rel = clean(name.split("corpus/", 1)[1])
        dest = corpus / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(z.read(m))
        n += 1
print("recovered files:", n)

recovered files: 16
